In [ ]:
import pandas as pd
import numpy as np
import ast
from scipy.stats import wilcoxon

df_ml = pd.read_csv('resultados/resultados_ML.csv')
df_dl = pd.read_csv('resultados/resultados_DL.csv')
df_est = pd.read_csv('resultados/resultados_estatisticos.csv')
df_all = pd.concat([df_ml, df_dl, df_est], ignore_index=True)

df_produto = pd.read_csv('data/product.csv', parse_dates=['Date']).set_index('Date')
df_pais = pd.read_csv('data/country.csv', parse_dates=['Date']).set_index('Date')
df_cliente = pd.read_csv('data/customer.csv', parse_dates=['Date']).set_index('Date')

series_data = {
    'Produto': df_produto['Revenue'].values,
    'País': df_pais['Revenue'].values,
    'Cliente': df_cliente['Revenue'].values
}

all_metrics_results = {'Erro Absoluto - Produto': [], 'Erro Absoluto - País': [], 'Erro Absoluto - Cliente': []}

for index, row in df_all.iterrows():
    serie = row['serie']
    modelo = row['modelo']
    periodos = int(row['periodos_previstos'])
    
    # Converter string de lista para array numpy real
    y_pred = np.array(ast.literal_eval(row['valores_previstos']))
    
    # Pegar os últimos 'N' dias reais correspondendo ao conjunto de teste
    y_true = series_data[serie][-periodos:]
    
    # Calcular Erro Absoluto para cada ponto predito
    erro_absoluto = np.abs(y_true - y_pred)
    
    if serie == 'Produto':
        chave = 'Erro Absoluto - Produto'
    elif serie == 'País':
        chave = 'Erro Absoluto - País'
    elif serie == 'Cliente':
        chave = 'Erro Absoluto - Cliente'
    else:
        continue
        
    all_metrics_results[chave].append((modelo, erro_absoluto))

alpha = 0.05

for metric_name, model_scores_list in all_metrics_results.items():
    print(f"Analisando: {metric_name}")
    
    for i in range(len(model_scores_list)):
        for j in range(i + 1, len(model_scores_list)):
            model1_name, scores1 = model_scores_list[i]
            model2_name, scores2 = model_scores_list[j]

            print(f"\nComparando {model1_name} x {model2_name}:")

            if len(scores1) != len(scores2):
                print(f"\t- ERRO: Tamanhos das listas de scores são diferentes. ({len(scores1)} vs {len(scores2)})")
                continue
            if len(scores1) < 2:
                print("\t- ERRO: Não há scores suficientes para o teste.")
                continue

            try:
                stat, p_val = wilcoxon(scores1, scores2, zero_method='zsplit', correction=False)
            except ValueError as e:
                print(f"\t- AVISO: Teste de Wilcoxon não pôde ser calculado. ({e})")
                mean1 = np.mean(scores1)
                mean2 = np.mean(scores2)
                print(f"\t- Média Erro {model1_name}: {mean1:.4f}")
                print(f"\t- Média Erro {model2_name}: {mean2:.4f}")
                print("\t-> Sem conclusão estatística.")
                print("----------------------------------------")
                continue

            mean1 = np.mean(scores1)
            mean2 = np.mean(scores2)

            print(f"\t- Amostras (n): {len(scores1)}")
            print(f"\t- Média Erro {model1_name}: {mean1:.4f}")
            print(f"\t- Média Erro {model2_name}: {mean2:.4f}")
            print(f"\t- Estatística Z/W: {stat:.4f}")
            print(f"\t- Valor-p (p-value): {p_val:.4f}")

            if p_val < alpha:
                print(f"\t-> Resultado: Diferença ESTATISTICAMENTE SIGNIFICATIVA (p < {alpha}).")
                best_model = model1_name if mean1 < mean2 else model2_name
                print(f"\t-> Melhor modelo (pelo menor erro): {best_model}")
            else:
                print(f"\t-> Resultado: Sem diferença estatisticamente significativa (p >= {alpha}).")

            print("----------------------------------------")
